In [1]:
import pandas as pd
import numpy as np
import os
import warnings

In [2]:
RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

print("Raw path:", RAW_PATH)
print("Processed path:", PROCESSED_PATH)

Raw path: ../data/raw
Processed path: ../data/processed


In [3]:
expected_files = [
    "City.xlsx",
    "Continent.xlsx",
    "Country.xlsx",
    "Item.xlsx",
    "Mode.xlsx",
    "Region.xlsx",
    "Transaction.xlsx",
    "Type.xlsx",
    "Updated_Item.xlsx",
    "User.xlsx"
]

print("DATASET AVAILABILITY")
print("=" * 60)

for file in expected_files:
    path = os.path.join(RAW_PATH, file)

    if os.path.exists(path):
        print(f"✓ {file}")
    else:
        print(f"✗ {file} — NOT FOUND")

DATASET AVAILABILITY
✓ City.xlsx
✓ Continent.xlsx
✓ Country.xlsx
✓ Item.xlsx
✓ Mode.xlsx
✓ Region.xlsx
✓ Transaction.xlsx
✓ Type.xlsx
✓ Updated_Item.xlsx
✓ User.xlsx


In [4]:
datasets = {}

for file in expected_files:
    path = os.path.join(RAW_PATH, file)

    if os.path.exists(path):
        dataset_name = os.path.splitext(file)[0].lower()

        datasets[dataset_name] = pd.read_excel(path)

print("Datasets loaded:")
print(list(datasets.keys()))

Datasets loaded:
['city', 'continent', 'country', 'item', 'mode', 'region', 'transaction', 'type', 'updated_item', 'user']


In [5]:
dataset_overview = []

for name, df in datasets.items():

    dataset_overview.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing_Values": df.isna().sum().sum(),
        "Duplicate_Rows": df.duplicated().sum()
    })

dataset_overview = pd.DataFrame(dataset_overview)

display(
    dataset_overview.sort_values(
        "Rows",
        ascending=False
    )
)

,Dataset,Rows,Columns,Missing_Values,Duplicate_Rows
6,transaction,52930,7,0,0
9,user,33530,5,4,0
0,city,9143,3,1,0
8,updated_item,1698,5,0,0
2,country,165,3,0,0
3,item,30,5,0,0
5,region,22,3,0,0
7,type,17,2,0,0
1,continent,6,2,0,0
4,mode,6,2,0,0


In [6]:
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(f"DATASET: {name.upper()}")
    print("=" * 60)

    print("Rows:", len(df))
    print("Columns:")

    for column in df.columns:
        print(" -", column)


DATASET: CITY
Rows: 9143
Columns:
 - CityId
 - CityName
 - CountryId

DATASET: CONTINENT
Rows: 6
Columns:
 - ContinentId
 - Continent

DATASET: COUNTRY
Rows: 165
Columns:
 - CountryId
 - Country
 - RegionId

DATASET: ITEM
Rows: 30
Columns:
 - AttractionId
 - AttractionCityId
 - AttractionTypeId
 - Attraction
 - AttractionAddress

DATASET: MODE
Rows: 6
Columns:
 - VisitModeId
 - VisitMode

DATASET: REGION
Rows: 22
Columns:
 - Region
 - RegionId
 - ContinentId

DATASET: TRANSACTION
Rows: 52930
Columns:
 - TransactionId
 - UserId
 - VisitYear
 - VisitMonth
 - VisitMode
 - AttractionId
 - Rating

DATASET: TYPE
Rows: 17
Columns:
 - AttractionTypeId
 - AttractionType

DATASET: UPDATED_ITEM
Rows: 1698
Columns:
 - AttractionId
 - AttractionCityId
 - AttractionTypeId
 - Attraction
 - AttractionAddress

DATASET: USER
Rows: 33530
Columns:
 - UserId
 - ContinentId
 - RegionId
 - CountryId
 - CityId


In [7]:
data_dictionary = []

for name, df in datasets.items():

    for column in df.columns:

        data_dictionary.append({
            "Dataset": name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Missing_Count": df[column].isna().sum(),
            "Unique_Values": df[column].nunique()
        })

data_dictionary = pd.DataFrame(data_dictionary)

display(data_dictionary)

,Dataset,Column,Data_Type,Missing_Count,Unique_Values
0,city,CityId,int64,0,9143
1,city,CityName,object,1,8767
2,city,CountryId,int64,0,164
3,continent,ContinentId,int64,0,6
4,continent,Continent,object,0,6
5,country,CountryId,int64,0,165
6,country,Country,object,0,164
7,country,RegionId,int64,0,22
8,item,AttractionId,int64,0,30
9,item,AttractionCityId,int64,0,3


In [8]:
def standardize_columns(df):
    
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    return df


for name in datasets:

    datasets[name] = standardize_columns(
        datasets[name]
    )

print("Column names standardized.")

Column names standardized.


In [9]:
for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print(list(df.columns))


CITY
['cityid', 'cityname', 'countryid']

CONTINENT
['continentid', 'continent']

COUNTRY
['countryid', 'country', 'regionid']

ITEM
['attractionid', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress']

MODE
['visitmodeid', 'visitmode']

REGION
['region', 'regionid', 'continentid']

TRANSACTION
['transactionid', 'userid', 'visityear', 'visitmonth', 'visitmode', 'attractionid', 'rating']

TYPE
['attractiontypeid', 'attractiontype']

UPDATED_ITEM
['attractionid', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress']

USER
['userid', 'continentid', 'regionid', 'countryid', 'cityid']


In [10]:
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(f"{name.upper()}")
    print("=" * 60)

    for column in df.columns:

        unique_count = df[column].nunique()
        row_count = len(df)

        if unique_count == row_count:

            print(
                f"Potential unique key: "
                f"{column} "
                f"({unique_count:,} unique)"
            )


CITY
Potential unique key: cityid (9,143 unique)

CONTINENT
Potential unique key: continentid (6 unique)
Potential unique key: continent (6 unique)

COUNTRY
Potential unique key: countryid (165 unique)

ITEM
Potential unique key: attractionid (30 unique)
Potential unique key: attraction (30 unique)

MODE
Potential unique key: visitmodeid (6 unique)
Potential unique key: visitmode (6 unique)

REGION
Potential unique key: region (22 unique)
Potential unique key: regionid (22 unique)

TRANSACTION
Potential unique key: transactionid (52,930 unique)

TYPE
Potential unique key: attractiontypeid (17 unique)
Potential unique key: attractiontype (17 unique)

UPDATED_ITEM
Potential unique key: attractionid (1,698 unique)

USER
Potential unique key: userid (33,530 unique)


In [11]:
for name, df in datasets.items():

    missing = (
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )

    missing = missing[missing > 0]

    print("\n" + "=" * 60)
    print(f"{name.upper()} — MISSING VALUES")
    print("=" * 60)

    if len(missing) == 0:
        print("No missing values.")
    else:
        display(missing.to_frame("Missing_Count"))


CITY — MISSING VALUES


,Missing_Count
cityname,1



CONTINENT — MISSING VALUES
No missing values.

COUNTRY — MISSING VALUES
No missing values.

ITEM — MISSING VALUES
No missing values.

MODE — MISSING VALUES
No missing values.

REGION — MISSING VALUES
No missing values.

TRANSACTION — MISSING VALUES
No missing values.

TYPE — MISSING VALUES
No missing values.

UPDATED_ITEM — MISSING VALUES
No missing values.

USER — MISSING VALUES


,Missing_Count
cityid,4


In [12]:
duplicate_summary = []

for name, df in datasets.items():

    duplicate_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Duplicate_Rows": df.duplicated().sum()
    })

duplicate_summary = pd.DataFrame(
    duplicate_summary
)

display(duplicate_summary)

,Dataset,Rows,Duplicate_Rows
0,city,9143,0
1,continent,6,0
2,country,165,0
3,item,30,0
4,mode,6,0
5,region,22,0
6,transaction,52930,0
7,type,17,0
8,updated_item,1698,0
9,user,33530,0


In [13]:
transaction = datasets["transaction"]

print("Transaction shape:", transaction.shape)

display(transaction.head())

print("\nColumns:")
print(transaction.columns.tolist())

Transaction shape: (52930, 7)


,transactionid,userid,visityear,visitmonth,visitmode,attractionid,rating
0,3,70456,2022,10,2,640,5
1,8,7567,2022,10,4,640,5
2,9,79069,2022,10,3,640,5
3,10,31019,2022,10,3,640,3
4,15,43611,2022,10,2,640,3



Columns:
['transactionid', 'userid', 'visityear', 'visitmonth', 'visitmode', 'attractionid', 'rating']


In [14]:
user = datasets["user"]

print("User shape:", user.shape)

display(user.head())

print("\nColumns:")
print(user.columns.tolist())

User shape: (33530, 5)


,userid,continentid,regionid,countryid,cityid
0,14,5,20,155,220.0
1,16,3,14,101,3098.0
2,20,4,15,109,4303.0
3,23,1,4,22,154.0
4,25,3,14,101,3098.0



Columns:
['userid', 'continentid', 'regionid', 'countryid', 'cityid']


In [15]:
city = datasets["city"]

print("City shape:", city.shape)

display(city.head())

print("\nColumns:")
print(city.columns.tolist())

City shape: (9143, 3)


,cityid,cityname,countryid
0,0,-,0
1,1,Douala,1
2,2,South Region,1
3,3,N'Djamena,2
4,4,Kigali,3



Columns:
['cityid', 'cityname', 'countryid']


In [16]:

item = datasets["item"]
updated_item = datasets["updated_item"]

print("ITEM")
print("=" * 50)
print("Shape:", item.shape)
print("Columns:", item.columns.tolist())

print("\nUPDATED ITEM")
print("=" * 50)
print("Shape:", updated_item.shape)
print("Columns:", updated_item.columns.tolist())

ITEM
Shape: (30, 5)
Columns: ['attractionid', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress']

UPDATED ITEM
Shape: (1698, 5)
Columns: ['attractionid', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress']


In [17]:
print("Item shape:", item.shape)
print("Updated Item shape:", updated_item.shape)

print("\nItem columns:")
print(item.columns.tolist())

print("\nUpdated Item columns:")
print(updated_item.columns.tolist())

Item shape: (30, 5)
Updated Item shape: (1698, 5)

Item columns:
['attractionid', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress']

Updated Item columns:
['attractionid', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress']


In [18]:
common_columns = list(
    set(item.columns) &
    set(updated_item.columns)
)

print("Common columns:")
print(common_columns)

Common columns:
['attractiontypeid', 'attraction', 'attractionid', 'attractioncityid', 'attractionaddress']


In [19]:
type_df = datasets["type"]

print("Type shape:", type_df.shape)

display(type_df.head())

print("\nColumns:")
print(type_df.columns.tolist())

Type shape: (17, 2)


,attractiontypeid,attractiontype
0,2,Ancient Ruins
1,10,Ballets
2,13,Beaches
3,19,Caverns & Caves
4,34,Flea & Street Markets



Columns:
['attractiontypeid', 'attractiontype']


In [20]:
mode = datasets["mode"]

print("Mode shape:", mode.shape)

display(mode.head())

print("\nColumns:")
print(mode.columns.tolist())

Mode shape: (6, 2)


,visitmodeid,visitmode
0,0,-
1,1,Business
2,2,Couples
3,3,Family
4,4,Friends



Columns:
['visitmodeid', 'visitmode']


In [21]:
for name in ["continent", "country", "region"]:

    df = datasets[name]

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    display(df.head())


CONTINENT
Shape: (6, 2)
Columns: ['continentid', 'continent']


,continentid,continent
0,0,-
1,1,Africa
2,2,America
3,3,Asia
4,4,Australia & Oceania



COUNTRY
Shape: (165, 3)
Columns: ['countryid', 'country', 'regionid']


,countryid,country,regionid
0,0,-,0
1,1,Cameroon,1
2,2,Chad,1
3,3,Rwanda,1
4,4,Ethiopia,2



REGION
Shape: (22, 3)
Columns: ['region', 'regionid', 'continentid']


,region,regionid,continentid
0,-,0,0
1,Central Africa,1,1
2,East Africa,2,1
3,North Africa,3,1
4,Southern Africa,4,1


In [22]:
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    key_columns = [
        col for col in df.columns
        if (
            "id" in col
            or "year" in col
            or "month" in col
            or "mode" in col
            or "rating" in col
        )
    ]

    for column in key_columns:

        print(
            f"{column}: "
            f"{df[column].nunique():,} unique | "
            f"{df[column].isna().sum():,} missing"
        )


CITY
cityid: 9,143 unique | 0 missing
countryid: 164 unique | 0 missing

CONTINENT
continentid: 6 unique | 0 missing

COUNTRY
countryid: 165 unique | 0 missing
regionid: 22 unique | 0 missing

ITEM
attractionid: 30 unique | 0 missing
attractioncityid: 3 unique | 0 missing
attractiontypeid: 17 unique | 0 missing

MODE
visitmodeid: 6 unique | 0 missing
visitmode: 6 unique | 0 missing

REGION
regionid: 22 unique | 0 missing
continentid: 6 unique | 0 missing

TRANSACTION
transactionid: 52,930 unique | 0 missing
userid: 33,530 unique | 0 missing
visityear: 10 unique | 0 missing
visitmonth: 12 unique | 0 missing
visitmode: 5 unique | 0 missing
attractionid: 30 unique | 0 missing
rating: 5 unique | 0 missing

TYPE
attractiontypeid: 17 unique | 0 missing

UPDATED_ITEM
attractionid: 1,698 unique | 0 missing
attractioncityid: 417 unique | 0 missing
attractiontypeid: 22 unique | 0 missing

USER
userid: 33,530 unique | 0 missing
continentid: 5 unique | 0 missing
regionid: 22 unique | 0 missing
co

In [23]:
def check_relationship(
    left_df,
    right_df,
    left_key,
    right_key
):

    left_values = set(
        left_df[left_key]
        .dropna()
        .unique()
    )

    right_values = set(
        right_df[right_key]
        .dropna()
        .unique()
    )

    unmatched = left_values - right_values

    print("=" * 60)
    print("RELATIONSHIP CHECK")
    print("=" * 60)

    print("Left key :", left_key)
    print("Right key:", right_key)

    print("Left unique keys :", len(left_values))
    print("Right unique keys:", len(right_values))
    print("Unmatched keys   :", len(unmatched))

    if len(unmatched) > 0:
        print("\nFirst unmatched values:")
        print(list(unmatched)[:20])

    return unmatched

In [24]:
transaction_user_unmatched = check_relationship(
    transaction,
    user,
    "userid",
    "userid"
)

RELATIONSHIP CHECK
Left key : userid
Right key: userid
Left unique keys : 33530
Right unique keys: 33530
Unmatched keys   : 0


In [25]:
user_city_unmatched = check_relationship(
    user,
    city,
    "cityid",
    "cityid"
)

RELATIONSHIP CHECK
Left key : cityid
Right key: cityid
Left unique keys : 5545
Right unique keys: 9143
Unmatched keys   : 0


In [26]:
transaction_item_unmatched = check_relationship(
    transaction,
    item,
    "attractionid",
    "attractionid"
)

RELATIONSHIP CHECK
Left key : attractionid
Right key: attractionid
Left unique keys : 30
Right unique keys: 30
Unmatched keys   : 0


In [27]:
city_country_unmatched = check_relationship(
    city,
    datasets["country"],
    "countryid",
    "countryid"
)

RELATIONSHIP CHECK
Left key : countryid
Right key: countryid
Left unique keys : 164
Right unique keys: 165
Unmatched keys   : 0


In [28]:
country_region_unmatched = check_relationship(
    datasets["country"],
    datasets["region"],
    "regionid",
    "regionid"
)

RELATIONSHIP CHECK
Left key : regionid
Right key: regionid
Left unique keys : 22
Right unique keys: 22
Unmatched keys   : 0


In [29]:
region_continent_unmatched = check_relationship(
    datasets["region"],
    datasets["continent"],
    "continentid",
    "continentid"
)

RELATIONSHIP CHECK
Left key : continentid
Right key: continentid
Left unique keys : 6
Right unique keys: 6
Unmatched keys   : 0


In [30]:
transaction_user_check = check_relationship(
    datasets["transaction"],
    datasets["user"],
    "userid",
    "userid"
)

RELATIONSHIP CHECK
Left key : userid
Right key: userid
Left unique keys : 33530
Right unique keys: 33530
Unmatched keys   : 0


In [31]:
user_city_check = check_relationship(
    datasets["user"],
    datasets["city"],
    "cityid",
    "cityid"
)

RELATIONSHIP CHECK
Left key : cityid
Right key: cityid
Left unique keys : 5545
Right unique keys: 9143
Unmatched keys   : 0


In [32]:
user_missing = datasets["user"][
    datasets["user"].isna().any(axis=1)
]

print("Users containing missing values:")
print("Number of affected records:", len(user_missing))

display(user_missing)

Users containing missing values:
Number of affected records: 4


,userid,continentid,regionid,countryid,cityid
2279,7175,5,17,135,NaN
6027,17595,1,4,22,NaN
21303,56972,5,17,135,NaN
25494,67461,5,17,135,NaN


In [33]:
transaction_item_check = check_relationship(
    datasets["transaction"],
    datasets["updated_item"],
    "attractionid",
    "attractionid"
)

RELATIONSHIP CHECK
Left key : attractionid
Right key: attractionid
Left unique keys : 30
Right unique keys: 1698
Unmatched keys   : 0


In [34]:
item_ids = set(
    datasets["item"]["attractionid"]
)

updated_item_ids = set(
    datasets["updated_item"]["attractionid"]
)

print("Item attractions:", len(item_ids))
print("Updated Item attractions:", len(updated_item_ids))

print("\nIDs in Item but not Updated_Item:")
print(sorted(item_ids - updated_item_ids))

print("\nIDs in Updated_Item but not Item:")
print(
    sorted(updated_item_ids - item_ids)[:30]
)

updated_item_ids = set(
    datasets["updated_item"]["attractionid"]
)

print("Item attractions:", len(item_ids))
print("Updated Item attractions:", len(updated_item_ids))

print("\nIDs in Item but not Updated_Item:")
print(sorted(item_ids - updated_item_ids))

print("\nIDs in Updated_Item but not Item:")
print(
    sorted(updated_item_ids - item_ids)[:30]
)

Item attractions: 30
Updated Item attractions: 1698

IDs in Item but not Updated_Item:
[]

IDs in Updated_Item but not Item:
[1298, 1299, 1300, 1301, 1302, 1303, 1304, 1305, 1306, 1307, 1308, 1309, 1310, 1311, 1312, 1313, 1314, 1315, 1316, 1317, 1318, 1319, 1320, 1321, 1322, 1323, 1324, 1325, 1326, 1327]
Item attractions: 30
Updated Item attractions: 1698

IDs in Item but not Updated_Item:
[]

IDs in Updated_Item but not Item:
[1298, 1299, 1300, 1301, 1302, 1303, 1304, 1305, 1306, 1307, 1308, 1309, 1310, 1311, 1312, 1313, 1314, 1315, 1316, 1317, 1318, 1319, 1320, 1321, 1322, 1323, 1324, 1325, 1326, 1327]


In [35]:
transaction_attraction_ids = set(
    datasets["transaction"]["attractionid"]
    if "attractionid" in datasets["transaction"].columns
    else datasets["transaction"]["AttractionId"]
)

covered_by_updated_item = (
    transaction_attraction_ids
    .intersection(updated_item_ids)
)

print(
    "Unique attractions in transactions:",
    len(transaction_attraction_ids)
)

print(
    "Found in Updated_Item:",
    len(covered_by_updated_item)
)

print(
    "Missing from Updated_Item:",
    len(
        transaction_attraction_ids
        - updated_item_ids
    )
)

Unique attractions in transactions: 30
Found in Updated_Item: 30
Missing from Updated_Item: 0


In [36]:
attraction_type_check = check_relationship(
    datasets["updated_item"],
    datasets["type"],
    "attractiontypeid",
    "attractiontypeid"
)

RELATIONSHIP CHECK
Left key : attractiontypeid
Right key: attractiontypeid
Left unique keys : 22
Right unique keys: 17
Unmatched keys   : 5

First unmatched values:
['Beach', 'Temple', 'Park', 'Market', 'Museum']


In [37]:
left_key = "attractioncityid" if "attractioncityid" in datasets["updated_item"].columns else "AttractionCityId"
right_key = "cityid" if "cityid" in datasets["city"].columns else "CityId"

attraction_city_check = check_relationship(
    datasets["updated_item"],
    datasets["city"],
    left_key,
    right_key
)

RELATIONSHIP CHECK
Left key : attractioncityid
Right key: cityid
Left unique keys : 417
Right unique keys: 9143
Unmatched keys   : 0


In [38]:
left_key = "countryid" if "countryid" in datasets["city"].columns else "CountryId"
right_key = "countryid" if "countryid" in datasets["country"].columns else "CountryId"

city_country_check = check_relationship(
    datasets["city"],
    datasets["country"],
    left_key,
    right_key
)

RELATIONSHIP CHECK
Left key : countryid
Right key: countryid
Left unique keys : 164
Right unique keys: 165
Unmatched keys   : 0


In [39]:
country_region_check = check_relationship(
    datasets["country"],
    datasets["region"],
    "regionid",
    "regionid"
)

RELATIONSHIP CHECK
Left key : regionid
Right key: regionid
Left unique keys : 22
Right unique keys: 22
Unmatched keys   : 0


In [40]:
left_key = "continentid" if "continentid" in datasets["region"].columns else "ContinentId"
right_key = "continentid" if "continentid" in datasets["continent"].columns else "ContinentId"

region_continent_check = check_relationship(
    datasets["region"],
    datasets["continent"],
    left_key,
    right_key
)

RELATIONSHIP CHECK
Left key : continentid
Right key: continentid
Left unique keys : 6
Right unique keys: 6
Unmatched keys   : 0


In [41]:
print("Transaction VisitMode values:")
transaction_visit_mode = datasets["transaction"].get("visitmode", datasets["transaction"].get("VisitMode"))
print("Transaction VisitMode values:")
print(
    sorted(
        pd.Series(transaction_visit_mode)
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nMode table:")
display(datasets["mode"])

print("\nMode table:")
display(datasets["mode"])

Transaction VisitMode values:
Transaction VisitMode values:
[1, 2, 3, 4, 5]

Mode table:


,visitmodeid,visitmode
0,0,-
1,1,Business
2,2,Couples
3,3,Family
4,4,Friends
5,5,Solo



Mode table:


,visitmodeid,visitmode
0,0,-
1,1,Business
2,2,Couples
3,3,Family
4,4,Friends
5,5,Solo


In [42]:
transaction = datasets["transaction"]

print("Unique ratings:")
transaction = datasets["transaction"]

print("Unique ratings:")
print(
    sorted(
        transaction["rating"]
        .dropna()
        .unique()
    )
)

print("\nRating distribution:")
display(
    transaction["rating"]
    .value_counts()
    .sort_index()
)

print("\nRating distribution:")
display(
    transaction["rating"]
    .value_counts()
    .sort_index()
)

Unique ratings:
Unique ratings:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Rating distribution:


rating
1     1263
2     2035
3     7730
4    17966
5    23936
Name: count, dtype: int64


Rating distribution:


rating
1     1263
2     2035
3     7730
4    17966
5    23936
Name: count, dtype: int64

In [43]:
print("Visit years:")
print(
    sorted(
        transaction["visityear"]
        .dropna()
        .unique()
    )
)

print("\nNumber of unique years:",
      transaction["visityear"].nunique())

print("\nNumber of unique years:",
      transaction["visityear"].nunique())

Visit years:
[np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Number of unique years: 10

Number of unique years: 10


In [44]:
print("Visit months:")
print(
    sorted(
        transaction["visitmonth"]
        .dropna()
        .unique()
    )
)

print("\nNumber of unique months:",
      transaction["visitmonth"].nunique())

Visit months:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

Number of unique months: 12
